# 09 — Evaluation and Testing

A lightweight evaluation framework: keyword checks, LLM-as-judge scoring, and batch evaluation.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Test Dataset and Helpers

In [ ]:
TEST_CASES = [
    {"input": "What is the capital of France?", "expected_keywords": ["paris"], "category": "factual"},
    {"input": "Explain photosynthesis in one sentence.", "expected_keywords": ["light", "energy", "plant"], "category": "explanation"},
    {"input": "What is 15 * 23?", "expected_keywords": ["345"], "category": "math"},
    {"input": "List three programming languages.", "expected_keywords": [], "category": "listing"},
    {"input": "Is the Earth flat?", "expected_keywords": ["no", "not flat", "sphere", "round", "oblate"], "category": "factual"},
]

def keyword_check(response: str, keywords: list[str]) -> bool:
    if not keywords:
        return True
    return any(kw.lower() in response.lower() for kw in keywords)

class QualityScore(BaseModel):
    score: int = Field(description="Quality score from 1 to 5")
    reasoning: str = Field(description="Brief explanation for the score")

## Run Evaluation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = ChatPromptTemplate.from_template("Answer this question concisely:\n{question}") | llm | StrOutputParser()
judge_prompt = ChatPromptTemplate.from_template(
    "Rate this answer on a scale of 1-5 for correctness and helpfulness.\n\n"
    "Question: {question}\nAnswer: {answer}\n\nScore 1 = wrong/unhelpful, 5 = perfect."
)
judge_chain = judge_prompt | judge.with_structured_output(QualityScore)

results = []
for i, test in enumerate(TEST_CASES, 1):
    response = qa_chain.invoke({"question": test["input"]})
    kw_pass = keyword_check(response, test["expected_keywords"])
    score = judge_chain.invoke({"question": test["input"], "answer": response})
    results.append({"kw_pass": kw_pass, "score": score.score, "category": test["category"]})
    status = "PASS" if kw_pass and score.score >= 3 else "FAIL"
    print(f"[{status}] Test {i}: {test['input']}")
    print(f"  Response: {response[:80]}...")
    print(f"  Keywords: {'PASS' if kw_pass else 'FAIL'} | Judge: {score.score}/5 — {score.reasoning}\n")

## Summary

In [ ]:
total = len(results)
passed = sum(1 for r in results if r["kw_pass"] and r["score"] >= 3)
avg_score = sum(r["score"] for r in results) / total
print(f"Passed: {passed}/{total} ({100*passed//total}%)")
print(f"Average judge score: {avg_score:.1f}/5")